In [ ]:
# importar librerias 
%pip install duckdb 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 1.7 MB/s  0:00:09m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb

In [18]:
#importar dataset
# DuckDB buscará el archivo en la misma carpeta donde está corriendo el Notebook
ventas_tecnologia = duckdb.sql("SELECT * FROM 'ventas_tecnologia.csv'").show()


┌──────────┬───────────────────────────┬─────────────┬─────────────────┬──────────┬────────────┬───────────┐
│ id_venta │         producto          │  categoria  │ precio_unitario │ cantidad │   fecha    │   pais    │
│  int64   │          varchar          │   varchar   │     double      │  int64   │    date    │  varchar  │
├──────────┼───────────────────────────┼─────────────┼─────────────────┼──────────┼────────────┼───────────┤
│        1 │ Laptop Lenovo IdeaPad 5   │ Laptops     │           750.0 │        3 │ 2026-01-08 │ Colombia  │
│        2 │ iPhone 15                 │ Smartphones │           899.0 │        5 │ 2026-01-12 │ México    │
│        3 │ Mouse Logitech M185       │ Accesorios  │            18.5 │       40 │ 2026-01-15 │ Colombia  │
│        4 │ Samsung Galaxy S24        │ Smartphones │           799.0 │        4 │ 2026-01-20 │ Chile     │
│        5 │ Audífonos Sony WH-1000XM5 │ Audio       │           349.0 │        6 │ 2026-02-02 │ Argentina │
│        6 │ iPad A

In [6]:
# 2. Selección Simple: Escribe una consulta que devuelva solo los nombres de los productos y su precio, ordenados de forma alfabética por nombre.

duckdb.sql("SELECT producto, precio_unitario FROM 'ventas_tecnologia.csv' ORDER BY producto ASC")

┌───────────────────────────┬─────────────────┐
│         producto          │ precio_unitario │
│          varchar          │     double      │
├───────────────────────────┼─────────────────┤
│ Audífonos Sony WH-1000XM5 │           349.0 │
│ Audífonos Sony WH-1000XM5 │           349.0 │
│ Cable USB-C 2m            │             9.9 │
│ JBL Flip 6                │           129.0 │
│ Laptop HP Pavilion 15     │           680.0 │
│ Laptop Lenovo IdeaPad 5   │           750.0 │
│ MacBook Air M3            │          1199.0 │
│ MacBook Air M3            │          1199.0 │
│ Mouse Logitech M185       │            18.5 │
│ Mouse Logitech M185       │            18.5 │
│ Samsung Galaxy S24        │           799.0 │
│ Samsung Galaxy S24        │           799.0 │
│ Samsung Galaxy Tab S9     │           649.0 │
│ Teclado Mecánico Redragon │            45.0 │
│ Teclado Mecánico Redragon │            45.0 │
│ Xiaomi Redmi Note 13      │           249.0 │
│ iPad Air                  │           

In [9]:
# 3. Filtrado Crítico: Obtén todas las ventas realizadas en el país 'Colombia' donde el precio_unitario sea mayor a 500.
duckdb.sql("SELECT * FROM 'ventas_tecnologia.csv' WHERE pais = 'Colombia' AND precio_unitario > 500 ")

┌──────────┬─────────────────────────┬─────────────┬─────────────────┬──────────┬────────────┬──────────┐
│ id_venta │        producto         │  categoria  │ precio_unitario │ cantidad │   fecha    │   pais   │
│  int64   │         varchar         │   varchar   │     double      │  int64   │    date    │ varchar  │
├──────────┼─────────────────────────┼─────────────┼─────────────────┼──────────┼────────────┼──────────┤
│        1 │ Laptop Lenovo IdeaPad 5 │ Laptops     │           750.0 │        3 │ 2026-01-08 │ Colombia │
│       10 │ Samsung Galaxy Tab S9   │ Tablets     │           649.0 │        2 │ 2026-03-09 │ Colombia │
│       20 │ Samsung Galaxy S24      │ Smartphones │           799.0 │        6 │ 2026-06-03 │ Colombia │
└──────────┴─────────────────────────┴─────────────┴─────────────────┴──────────┴────────────┴──────────┘

In [ ]:
# 4. Búsqueda de Nulos: Identifica si hay registros donde la categoria sea NULL (compañeros del equipo de ingeniería a veces olvidan este campo).
duckdb.sql("SELECT * FROM 'ventas_tecnologia.csv' WHERE categoria IS NULL")

┌──────────┬──────────┬───────────┬─────────────────┬──────────┬───────┬─────────┐
│ id_venta │ producto │ categoria │ precio_unitario │ cantidad │ fecha │  pais   │
│  int64   │ varchar  │  varchar  │     double      │  int64   │ date  │ varchar │
└──────────┴──────────┴───────────┴─────────────────┴──────────┴───────┴─────────┘
                                      0 rows                                    

In [16]:
''' 
5. Análisis de Rendimiento (Agregación):
    * Calcula el total de ingresos (cantidad * precio_unitario) por cada categoria.
    * Nombra a la columna resultante como ingresos_totales usando un alias.
'''
duckdb.sql("SELECT categoria, " \
    "SUM(cantidad * precio_unitario) AS ingresos_totales" \
    " FROM 'ventas_tecnologia.csv' " \
    "GROUP BY categoria")

┌─────────────┬──────────────────┐
│  categoria  │ ingresos_totales │
│   varchar   │      double      │
├─────────────┼──────────────────┤
│ Laptops     │          10965.0 │
│ Audio       │           4689.0 │
│ Tablets     │           6090.0 │
│ Smartphones │          22513.0 │
│ Accesorios  │           4022.0 │
└─────────────┴──────────────────┘

In [21]:
# 6. Filtro de Élite (HAVING): Muestra solo las categorías que han generado más de $10,000 en ingresos totales.

duckdb.sql("SELECT categoria,"
    "SUM(cantidad * precio_unitario) AS ingresos_totales " \
    "FROM 'ventas_tecnologia.csv' " \
    "GROUP BY categoria " \
    "HAVING SUM(cantidad * precio_unitario)>10000")

┌─────────────┬──────────────────┐
│  categoria  │ ingresos_totales │
│   varchar   │      double      │
├─────────────┼──────────────────┤
│ Laptops     │          10965.0 │
│ Smartphones │          22513.0 │
└─────────────┴──────────────────┘